In [ ]:
import sys
print(sys.executable)

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./vectorstores/alice")
collection = client.get_collection("alice")

# 查看資料庫內所有資料
results = collection.get(include=["documents", "metadatas"])

print(f"共有 {len(results['ids'])} 筆資料")
for i, (id_, doc, meta) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
    print(f"\n[{i+1}] id: {id_}")
    print(f"     type: {meta.get('type')}")
    print(f"     source: {meta.get('source')}")
    print(f"     document: {doc[:100]}")

In [ ]:
import os
from dotenv import load_dotenv

# 從 .env 載入機密金鑰，避免硬編碼在程式碼中
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

# 機密金鑰由 .env 提供
for _key in ("LANGCHAIN_API_KEY", "GEMINI_API_KEY_PRIMARY", "GEMINI_API_KEY_SECONDARY"):
    if not os.environ.get(_key):
        raise RuntimeError(f"缺少環境變數 {_key}，請在 .env 檔中設定")

# Gemini Flash 有兩把金鑰以分散用量限制：
#   主要金鑰 (預設使用) / 備用金鑰 (主要金鑰用完額度時可切換)
gemini_api_key_primary = os.environ["GEMINI_API_KEY_PRIMARY"]
gemini_api_key_secondary = os.environ["GEMINI_API_KEY_SECONDARY"]

# 預設使用主要金鑰
active_gemini_api_key = gemini_api_key_primary

import torch
import open_clip
import chromadb
from langchain_google_genai import ChatGoogleGenerativeAI

# 載入 CLIP 模型（與輸入程式相同）
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()

# 用 CLIP embed 問題文字
def embed_text(texts: list[str]):
    tokens = tokenizer(texts)
    with torch.no_grad():
        features = model.encode_text(tokens)
        features /= features.norm(dim=-1, keepdim=True)
    return features.numpy().tolist()

# 連接對應使用者的資料庫
user_name = "alice"
client = chromadb.PersistentClient(path=f"./vectorstores/{user_name}")
vector_collection = client.get_collection(user_name)  # collection 名稱 = user_name

# Gemini 生成模型（預設使用主要金鑰，可改成 gemini_api_key_secondary 切換備用金鑰）
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    google_api_key=active_gemini_api_key
)

def rag_query(user_question, top_k=5):
    # Step 1: 用 CLIP embed 問題
    q_embedding = embed_text([user_question])[0]

    # Step 2: 搜尋相關文件
    results = vector_collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k
    )

    # Step 3: 取出文件內容
    docs = results["documents"][0]
    context = "\n---\n".join(docs)

    # Step 4: 組成 Prompt 呼叫 Gemini
    prompt = f"""你是一個問答助手，請根據以下提供的資料盡量完整且詳細的回答問題。
如果資料中沒有相關資訊，請回答「參考資料中沒有記載相關訊息」，不要自行編造。
注意: 1.請勿使用粗體 2.請勿套用latex格式 3.你得到的資料是基於原始資料的敘述，若敘述中含有使用者提問的項目，請將其統整後再回答。

【參考資料】
{context}

【使用者問題】
{user_question}

【回答】"""

    response = llm.invoke(prompt)
    return response.content

input_question = input('輸入問題: ')
answer = rag_query(
    user_question=input_question,
    top_k=10
)
print(answer)